<a href="https://colab.research.google.com/github/tejaswinisamanta/Tokens-to-Transformers-GenAI-x-LLM-Course-/blob/main/W1D1_NLP_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 · Day 1 · Task 3 — Run an NLP pipeline on a paragraph

**From Tokens to Transformers** · Consulting & Analytics Club, IIT Guwahati

You just watched two videos on what NLP is and what a pipeline looks like. This notebook makes that concrete: you will push one paragraph through a real pipeline and look at what comes out at every stage.

**Time:** about 30 minutes. **What you need:** a Google account (this runs in Colab). Nothing to install on your laptop.

**By the end you will be able to say, with evidence:**

1. what a *token* is, and why "don't" is not one token,
2. what a pipeline *stage* is, and what the tagger, parser and entity recogniser each add,
3. why a rule-based tool like spaCy is fast and predictable, and where it breaks — which is exactly the gap that language models fill in Week 2.

> **How to use this notebook:** run every cell in order (`Shift + Enter`). Cells marked **✎ Your turn** need you to edit or write a line. Do not skip them; they are the whole point.

## 0 · Setup (run once, ~1 min)

In [3]:
# spaCy is an industrial NLP library. It runs a fixed pipeline of stages over text.
# The small English model is ~13 MB; it is already installed on Colab, so this is quick.
!pip -q install spacy
!python -m spacy download en_core_web_sm -q

import spacy
nlp = spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__)
print("pipeline stages, in order:", nlp.pipe_names)

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy 3.8.16
pipeline stages, in order: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


Read that last line. Those names are the **stages** of the pipeline, in the order the text passes through them:

| stage | what it adds |
|---|---|
| `tok2vec` | turns each token into a vector the later stages use (you will meet vectors properly tomorrow) |
| `tagger` | part-of-speech tag for every token: noun, verb, adjective… |
| `parser` | the grammatical tree: which word is the subject, which is the object, what modifies what |
| `attribute_ruler` | small rule-based fixes on top of the tagger |
| `lemmatizer` | the dictionary form of each word: *running → run*, *better → good* |
| `ner` | named-entity recogniser: people, organisations, places, dates, money… |

**Tokenisation is not in the list** because it runs before all of them. Everything downstream works on tokens, never on raw text. Keep that in mind all week.

## 1 · One paragraph in, a `Doc` out

In [30]:
text = (
    "The Consulting & Analytics Club at IIT Guwahati is running a free four-week course "
    "on large language models from 28 September 2026. A free Colab GPU has about 15,000 MB "
    "of memory, so nobody needs to buy hardware. Andrej Karpathy's videos and Google Colab "
    "notebooks don't cost anything, which is why the course costs ₹0."
)

doc = nlp(text)          # this single call runs every stage in the pipeline
type(doc), len(doc)      # a Doc object, and how many tokens it holds

(spacy.tokens.doc.Doc, 65)

`nlp(text)` gave you a `Doc`. It looks like a string but it is a **sequence of tokens**, each carrying everything the pipeline computed for it. `len(doc)` is the number of tokens, not characters or words.

✎ **Your turn:** before running the next cell, guess the number of tokens by counting the words in `text` by eye. Write your guess here: `__56__`. Then run it.

In [5]:
print("characters :", len(text))
print("words (split on spaces):", len(text.split()))
print("tokens (spaCy)         :", len(doc))

characters : 319
words (split on spaces): 55
tokens (spaCy)         : 65


The three numbers differ. The gap between *words split on spaces* and *tokens* is the tokenizer doing its job. Let us see exactly where it split.

## 2 · Stage 0: tokenization

In [6]:
for i, token in enumerate(doc):
    print(f"{i:>3}  {token.text!r:<16} whitespace_after={token.whitespace_!r}")

  0  'The'            whitespace_after=' '
  1  'Consulting'     whitespace_after=' '
  2  '&'              whitespace_after=' '
  3  'Analytics'      whitespace_after=' '
  4  'Club'           whitespace_after=' '
  5  'at'             whitespace_after=' '
  6  'IIT'            whitespace_after=' '
  7  'Guwahati'       whitespace_after=' '
  8  'is'             whitespace_after=' '
  9  'running'        whitespace_after=' '
 10  'a'              whitespace_after=' '
 11  'free'           whitespace_after=' '
 12  'four'           whitespace_after=''
 13  '-'              whitespace_after=''
 14  'week'           whitespace_after=' '
 15  'course'         whitespace_after=' '
 16  'on'             whitespace_after=' '
 17  'large'          whitespace_after=' '
 18  'language'       whitespace_after=' '
 19  'models'         whitespace_after=' '
 20  'from'           whitespace_after=' '
 21  '28'             whitespace_after=' '
 22  'September'      whitespace_after=' '
 23  '2026'  

Look for these three things in the list above:

- **Punctuation is its own token.** The full stop after *2026* is a token; so is the ampersand in *Consulting & Analytics*.
- **Contractions are split.** *don't* became `do` + `n't`. The tokenizer knows English contractions by rule.
- **Numbers and symbols stay together when they should.** *15,000* is one token; *₹0* became `₹` + `0`; *Karpathy's* became `Karpathy` + `'s`. Scroll down in the output to find them.

None of this needed a model. spaCy's tokenizer is a set of rules: split on spaces, then apply prefix, suffix and infix rules, then check a list of exceptions like *don't*. That is why it is fast, and why it will do something surprising on text its rules were not written for.

✎ **Your turn:** edit the string below so that it contains (a) an email address, (b) a URL, (c) a hashtag, (d) a word with a hyphen. Run the cell and see which of those the tokenizer keeps whole and which it breaks up.

In [36]:
tricky = "Email tejaswini.2024.1705@mirandahouse.ac.in or visit https://mirandahouse.in — use #SheforHer for your week-1 post."

for token in nlp(tricky):
    print(repr(token.text), end="  ")

'Email'  'tejaswini.2024.1705@mirandahouse.ac.in'  'or'  'visit'  'https://mirandahouse.in'  '—'  'use'  '#'  'SheforHer'  'for'  'your'  'week-1'  'post'  '.'  

## 3 · Stage 1: part-of-speech tags

In [11]:
# .pos_ is the coarse tag (NOUN, VERB, ...). .tag_ is the fine-grained one (NNS, VBZ, ...).
# .lemma_ is the dictionary form.
print(f"{'token':<14}{'pos':<8}{'tag':<7}{'lemma'}")
print("-" * 40)
for token in doc[:28]:
    print(f"{token.text:<14}{token.pos_:<8}{token.tag_:<7}{token.lemma_}")

token         pos     tag    lemma
----------------------------------------
The           DET     DT     the
Consulting    PROPN   NNP    Consulting
&             CCONJ   CC     &
Analytics     PROPN   NNP    Analytics
Club          PROPN   NNP    Club
at            ADP     IN     at
IIT           PROPN   NNP    IIT
Guwahati      PROPN   NNP    Guwahati
is            AUX     VBZ    be
running       VERB    VBG    run
a             DET     DT     a
free          ADJ     JJ     free
four          NUM     CD     four
-             PUNCT   HYPH   -
week          NOUN    NN     week
course        NOUN    NN     course
on            ADP     IN     on
large         ADJ     JJ     large
language      NOUN    NN     language
models        NOUN    NNS    model
from          ADP     IN     from
28            NUM     CD     28
September     PROPN   NNP    September
2026          NUM     CD     2026
.             PUNCT   .      .
A             DET     DT     a
free          ADJ     JJ     free
Cola

Two things to check against your intuition:

1. *running* is tagged `VERB` with lemma `run`; *course* is `NOUN`. The tagger decided that from context, not from a dictionary — *running* can be a noun ("a running of the bulls").
2. `spacy.explain()` decodes any tag you do not recognise. Try it.

In [12]:
for tag in ["PROPN", "AUX", "ADP", "NNS", "VBZ"]:
    print(f"{tag:<6} → {spacy.explain(tag)}")

PROPN  → proper noun
AUX    → auxiliary
ADP    → adposition
NNS    → noun, plural
VBZ    → verb, 3rd person singular present


✎ **Your turn:** the sentence below has the word *book* twice, once as a verb and once as a noun. Run it. The tagger gets the first one **wrong** — an imperative verb at the start of a sentence looks like a noun to it. Now edit the sentence so that both are tagged correctly without changing either *book*. (Hint: give the tagger one more word of context.)

In [13]:
s = "Book a room before you read the book."
print([(t.text, t.pos_) for t in nlp(s)])

[('Book', 'NOUN'), ('a', 'DET'), ('room', 'NOUN'), ('before', 'SCONJ'), ('you', 'PRON'), ('read', 'VERB'), ('the', 'DET'), ('book', 'NOUN'), ('.', 'PUNCT')]


## 4 · Stage 2: the dependency parse

The parser draws a tree over each sentence: every token gets a **head** (the word it depends on) and a **relation** to it (`nsubj` = subject, `dobj` = direct object, `amod` = adjective modifier…). This is how a rule-based system answers "who did what to whom".

In [14]:
sent = list(doc.sents)[1]          # the second sentence
print(sent.text, "\n")
print(f"{'token':<12}{'relation':<10}{'head'}")
print("-" * 32)
for token in sent:
    print(f"{token.text:<12}{token.dep_:<10}{token.head.text}")

A free Colab GPU has about 15,000 MB of memory, so nobody needs to buy hardware. 

token       relation  head
--------------------------------
A           det       GPU
free        amod      GPU
Colab       compound  GPU
GPU         nsubj     has
has         ccomp     needs
about       advmod    15,000
15,000      nummod    MB
MB          dobj      has
of          prep      MB
memory      pobj      of
,           punct     needs
so          advmod    needs
nobody      nsubj     needs
needs       ROOT      needs
to          aux       buy
buy         xcomp     needs
hardware    dobj      buy
.           punct     needs


In [15]:
# Render the tree. Scroll sideways; arrows point from head to dependent.
from spacy import displacy
displacy.render(sent, style="dep", jupyter=True, options={"distance": 90, "compact": True})

Find the **root** of the sentence (the token whose head is itself). Everything hangs off it. This sentence has two clauses joined by *so*, and the parser picked `needs` as the root, with `has` hanging off it as a `ccomp`. Trace `GPU → has → needs` and `hardware → buy → needs` in the tree.

✎ **Your turn:** write a two-line function that, given a sentence, returns `(subject, verb, object)` using only `token.dep_` and `token.head`. Test it on the sentence in the cell. Then test it on a passive sentence — *"The course was designed by students."* — and write one line on what went wrong.

In [41]:
def svo(sentence):
    d = nlp(sentence)
    subj = next((t.text for t in d if t.dep_ == "nsubj"), None)
    verb = next((t.text for t in d if t.dep_ == "ROOT"), None)
    obj  = next((t.text for t in d if t.dep_ in ("dobj", "pobj")), None)
    return subj, verb, obj

print(svo("Students build a chatbot in week three."))
print(svo("The course was designed by students."))   # ← what happens here, and why?

('Students', 'build', 'chatbot')
(None, 'designed', 'students')


## 5 · Stage 3: named entities

In [31]:
print(f"{'entity':<40}{'label':<8}explanation")
print("-" * 80)
for ent in doc.ents:
    print(f"{ent.text:<40}{ent.label_:<8}{spacy.explain(ent.label_)}")

entity                                  label   explanation
--------------------------------------------------------------------------------
The Consulting & Analytics Club         ORG     Companies, agencies, institutions, etc.
IIT Guwahati                            ORG     Companies, agencies, institutions, etc.
four-week                               DATE    Absolute or relative dates or periods
28 September 2026                       DATE    Absolute or relative dates or periods
about 15,000                            CARDINALNumerals that do not fall under another type
Andrej Karpathy's                       PERSON  People, including fictional
Google Colab                            ORG     Companies, agencies, institutions, etc.
0                                       MONEY   Monetary values, including unit


In [32]:
displacy.render(doc, style="ent", jupyter=True)

Read the entity list critically. Most of these are right; at least two are off. *four-week* is not really a `DATE`, and the model labelled `0` as `MONEY` but dropped the `₹` that made it money. That is the honest state of a small statistical model: good on the common cases, shaky at the edges.

✎ **Your turn:** answer these three in the markdown cell below, in your own words.

1. Which entities did the model get wrong or only half-right? (Look at *four-week*, the currency, and whether *Andrej Karpathy's* should include the *'s*.)
2. Change *IIT Guwahati* to *IITG* in `text`, re-run sections 1 and 5. Does the entity survive?
3. Add a sentence about a fictional person and a fictional company. Does the model label them anyway? What does that tell you about how it decides?

**My answers**

1. Four-week is a period of time. It's not a date. The name Andrej Karpathy should be without 's.  
2. After changing IIT Guwahati to IITG, the entity does not survive.
3. After adding the sentence 'Jude Duarte is the head of the faerie', the model labelled Jude as a person but not the faerie. But changing the first letter of faerie to Faerie, the model labelled it as FAC (Buildings, airports, highways, bridges, etc.).

## 6 · What a pipeline is for: one small, useful thing

Everything above is inspection. Pipelines exist to do work. Here is the smallest useful thing you can build with what you have: pull every *(organisation, date)* pair out of a text — the kind of extraction that powers a news-monitoring or a résumé-screening tool.

In [34]:
def orgs_and_dates(text):
    d = nlp(text)
    orgs  = sorted({e.text for e in d.ents if e.label_ == "ORG"})
    dates = sorted({e.text for e in d.ents if e.label_ == "DATE"})
    return {"organisations": orgs, "dates": dates}

news = (
    "Reliance Industries and Tata Consultancy Services announced a joint AI lab on 14 March 2026. "
    "The lab, based in Bengaluru, will open next quarter and hire 400 engineers by December."
)

orgs_and_dates(news)

{'organisations': ['Reliance Industries', 'Tata Consultancy Services'],
 'dates': ['14 March 2026', 'December', 'next quarter']}

✎ **Your turn:** paste three or four sentences from any news article you read today into `my_text` and run `orgs_and_dates(my_text)`. Count: how many organisations did it find, how many did it miss, how many did it invent? Write the three numbers here: `found _1_ · missed _1_ · invented _0_`.

In [35]:
my_text = """
    A routine flight from Dubai to Tel Aviv turned into a terrifying mid air emergency on Wednesday after an apparent violent confrontation between the pilots of a flydubai aircraft.
    Flight FZ1073, carrying around 180 people, was forced to divert to Saudi Arabia after transmitting emergency signals, including the international code associated with unlawful interference.

"""
orgs_and_dates(my_text)

{'organisations': [], 'dates': ['Wednesday']}

## 7 · Why this matters for the rest of the course

You ran a **fixed pipeline**: tokenizer → tagger → parser → entity recogniser, each stage hand-designed, each producing a labelled output the next stage consumes. It is fast (this whole notebook ran in under a second of compute), it is inspectable (you saw every intermediate result), and it fails in predictable ways (passive voice, unseen names, tricky tokens).

A large language model is a very different pipeline: **tokenizer → embeddings → attention → prediction**, with nothing hand-designed after the tokenizer. Tomorrow you will look at the first two stages of that pipeline — how text becomes tokens, and how tokens become vectors — and by Friday you will trace one sentence through all four.

Keep two questions from today:

- *What did the tokenizer do to my text before anything else saw it?* — you will ask this about every model you ever use.
- *Where does the rule-based approach break?* — those breaks are the reason the field moved to learned representations.

**Before you close:** `File → Save a copy in Drive`. You will submit notebooks by link all course long, so get the habit now.

---
*Adapted for From Tokens to Transformers from the [codebasics nlp-tutorials](https://github.com/codebasics/nlp-tutorials) spaCy notebooks. Runs on the free Colab tier.*